In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

BASE = "/content/drive/MyDrive/NatureLM_project"

folders = ["code", "audio_files", "outputs", "hf_cache"]

for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("✅ Folders ready")

✅ Folders ready


In [3]:
%cd /content/drive/MyDrive/NatureLM_project/code

!git clone https://github.com/earthspecies/NatureLM-audio.git

/content/drive/MyDrive/NatureLM_project/code
fatal: destination path 'NatureLM-audio' already exists and is not an empty directory.


In [4]:
%cd /content/drive/MyDrive/NatureLM_project/code/NatureLM-audio

/content/drive/MyDrive/NatureLM_project/code/NatureLM-audio


In [1]:
!pip install \
transformers==4.44.2 \
tokenizers==0.19.1 \
peft==0.11.1 \
accelerate==0.33.0 \
librosa soundfile sentencepiece \
mir_eval resampy \
git+https://github.com/earthspecies/beans-zero.git \
--no-cache-dir

  Cloning https://github.com/earthspecies/beans-zero.git to /tmp/pip-req-build-v_68oced
  Running command git clone --filter=blob:none --quiet https://github.com/earthspecies/beans-zero.git /tmp/pip-req-build-v_68oced
  Resolved https://github.com/earthspecies/beans-zero.git to commit 31d4487ee6452ae6c31853d45fd38b7d4150372d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/NatureLM_project/hf_cache"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!python cli.py infer \
--cfg-path configs/inference.yml \
--audio-path "/content/drive/MyDrive/NatureLM_project/audio_files/batch_1" \
--query "Detect cane toad sounds" \
--output-path "/content/drive/MyDrive/NatureLM_project/outputs/batch_1.jsonl"

python3: can't open file '/content/cli.py': [Errno 2] No such file or directory


In [6]:
%cd /content/drive/MyDrive/NatureLM_project/code/NatureLM-audio

/content/drive/MyDrive/NatureLM_project/code/NatureLM-audio


In [7]:
from huggingface_hub import login
login()

In [8]:
batch = input("Which batch: ")

audio_folder = f"/content/drive/MyDrive/NatureLM_project/audio_files/batch_{batch}"
result_folder = "/content/drive/MyDrive/NatureLM_project/outputs"

import os
os.makedirs(result_folder, exist_ok=True)

!python cli.py infer \
--cfg-path configs/inference.yml \
--audio-path "{audio_folder}" \
--query "Detect cane toad sounds"

Which batch: 21
Traceback (most recent call last):
  File "/content/drive/MyDrive/NatureLM_project/code/NatureLM-audio/cli.py", line 7, in <module>
    from inference_web_app import main as app_fn
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1091, in get_code
  File "<frozen importlib._bootstrap_external>", line 1191, in get_data
KeyboardInterrupt
^C


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
%cd /content/drive/MyDrive/NatureLM_project/code/NatureLM-audio

/content/drive/MyDrive/NatureLM_project/code/NatureLM-audio


In [ ]:
import os
import time
import shutil
import subprocess

BASE = "/content/drive/MyDrive/NatureLM_project"
AUDIO_PATH = f"{BASE}/audio_files/all_files"
OUTPUT_PATH = f"{BASE}/outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# 🔥 Batch sizes to test
batch_sizes = [5, 10, 15, 20, 25, 30, 40, 50]

results = []

# Get all audio files
all_files = sorted([
    f for f in os.listdir(AUDIO_PATH)
    if f.endswith((".wav", ".mp3", ".flac", ".ogg"))
])

print(f"Total audio files found: {len(all_files)}")

for size in batch_sizes:
    print(f"\n🚀 Testing batch size: {size}")

    temp_batch = f"{BASE}/audio_files/temp_batch"
    os.makedirs(temp_batch, exist_ok=True)

    # clear temp folder
    for f in os.listdir(temp_batch):
        os.remove(os.path.join(temp_batch, f))

    # copy N files
    for f in all_files[:size]:
        shutil.copy(os.path.join(AUDIO_PATH, f), temp_batch)

    start = time.time()

    try:
        subprocess.run([
            "python", "cli.py", "infer",
            "--cfg-path", "configs/inference.yml",
            "--audio-path", temp_batch,
            "--query", "Detect cane toad sounds",
            "--output-path", f"{OUTPUT_PATH}/batch_{size}.jsonl"
        ], check=True)

        runtime = round(time.time() - start, 2)
        print(f"✅ SUCCESS ({runtime}s)")
        results.append((size, "SUCCESS", runtime))

    except:
        print("❌ FAILED")
        results.append((size, "FAILED", None))
        break

# save summary
with open(f"{OUTPUT_PATH}/batch_experiment.txt", "w") as f:
    for r in results:
        f.write(str(r) + "\n")

print("\n📊 Experiment completed and saved to Drive")

Total audio files found: 22006

🚀 Testing batch size: 5
✅ SUCCESS (93.81s)

🚀 Testing batch size: 10
✅ SUCCESS (89.61s)

🚀 Testing batch size: 15
✅ SUCCESS (92.92s)

🚀 Testing batch size: 20
✅ SUCCESS (96.29s)

🚀 Testing batch size: 25
✅ SUCCESS (99.52s)

🚀 Testing batch size: 30
✅ SUCCESS (102.75s)

🚀 Testing batch size: 40
✅ SUCCESS (108.88s)

🚀 Testing batch size: 50
✅ SUCCESS (114.94s)

📊 Experiment completed and saved to Drive


In [2]:
import os
import shutil
import subprocess
import time

BASE = "/content/drive/MyDrive/NatureLM_project"
AUDIO_PATH = f"{BASE}/audio_files/all_files"
OUTPUT_PATH = f"{BASE}/outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# 🔥 start + step (you can adjust)
start_size = 280
step = 50
max_limit = 22000   # safety stop

all_files = sorted([
    f for f in os.listdir(AUDIO_PATH)
    if f.endswith((".wav", ".mp3", ".flac", ".ogg"))
])

results = []

for size in range(start_size, max_limit + 1, step):

    print(f"\n🚀 Testing batch size: {size}")

    temp_batch = f"{BASE}/audio_files/temp_batch"
    os.makedirs(temp_batch, exist_ok=True)

    # clear temp
    for f in os.listdir(temp_batch):
        os.remove(os.path.join(temp_batch, f))

    # copy N files
    for f in all_files[:size]:
        shutil.copy(os.path.join(AUDIO_PATH, f), temp_batch)

    output_file = f"{OUTPUT_PATH}/batch_{size}.jsonl"

    start = time.time()

    try:
        subprocess.run([
            "python", "cli.py", "infer",
            "--cfg-path", "configs/inference.yml",
            "--audio-path", temp_batch,
            "--query", "Detect cane toad sounds",
            "--output-path", output_file
        ], check=True)

        runtime = round(time.time() - start, 2)

        print(f"✅ SUCCESS ({runtime}s)")
        results.append((size, "SUCCESS", runtime))

    except:
        print(f"❌ FAILED at {size}")
        results.append((size, "FAILED", None))
        break

# save experiment log
with open(f"{OUTPUT_PATH}/batch_test_results.txt", "w") as f:
    for r in results:
        f.write(str(r) + "\n")

print("\n📊 Experiment completed")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NatureLM_project/audio_files/all_files'

In [ ]:
import os
import shutil
import subprocess

BASE = "/content/drive/MyDrive/NatureLM_project"
AUDIO_PATH = f"{BASE}/audio_files/all_files"
OUTPUT_PATH = f"{BASE}/outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

CHUNK_SIZE = 50  # safe limit (can adjust later)

all_files = sorted([
    f for f in os.listdir(AUDIO_PATH)
    if f.endswith((".wav", ".mp3", ".flac", ".ogg"))
])

# detect last completed chunk
existing_chunks = sorted([
    int(f.split("_")[1].split(".")[0])
    for f in os.listdir(OUTPUT_PATH)
    if f.startswith("chunk_")
])

start_chunk = max(existing_chunks) + 1 if existing_chunks else 1

print(f"▶ Starting from chunk {start_chunk}")

# process continuously
chunk_idx = start_chunk

for i in range((start_chunk-1)*CHUNK_SIZE, len(all_files), CHUNK_SIZE):
    chunk = all_files[i:i+CHUNK_SIZE]

    print(f"\n🚀 Processing chunk {chunk_idx}")

    temp_batch = f"{BASE}/audio_files/temp_batch"
    os.makedirs(temp_batch, exist_ok=True)

    # clear temp
    for f in os.listdir(temp_batch):
        os.remove(os.path.join(temp_batch, f))

    # copy files
    for f in chunk:
        shutil.copy(os.path.join(AUDIO_PATH, f), temp_batch)

    output_file = f"{OUTPUT_PATH}/chunk_{chunk_idx}.jsonl"

    try:
        subprocess.run([
            "python", "cli.py", "infer",
            "--cfg-path", "configs/inference.yml",
            "--audio-path", temp_batch,
            "--query", "Detect cane toad sounds",
            "--output-path", output_file
        ], check=True)

        print(f"✅ Saved chunk {chunk_idx}")

    except:
        print(f"❌ Failed at chunk {chunk_idx}")
        break

    chunk_idx += 1

print("\n🎯 Finished (continuous processing)")

▶ Starting from chunk 1

🚀 Processing chunk 1
✅ Saved chunk 1

🚀 Processing chunk 2
✅ Saved chunk 2

🚀 Processing chunk 3
✅ Saved chunk 3

🚀 Processing chunk 4
✅ Saved chunk 4

🚀 Processing chunk 5
✅ Saved chunk 5

🚀 Processing chunk 6
✅ Saved chunk 6

🚀 Processing chunk 7
✅ Saved chunk 7

🚀 Processing chunk 8
✅ Saved chunk 8

🚀 Processing chunk 9
✅ Saved chunk 9

🚀 Processing chunk 10
✅ Saved chunk 10

🚀 Processing chunk 11
❌ Failed at chunk 11

🎯 Finished (continuous processing)
